# Alternative Promoter Expression Analysis

This notebook loads promoter-level raw counts and sample metadata, normalizes promoter- and gene-level counts, and compares promoter expression across cell types. Update the input file paths in the data-loading cells when running this notebook in a different environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)  # fixed seed for reproducibility

import os
DATA_DIR = os.environ.get("CHO_DATA_DIR", os.path.join("..", "..", "data"))  # repository data/ folder (notebook sits two levels below the repo root)


## Load Promoter-Level Raw Counts

Load the promoter-level count table. Each row represents a promoter, with promoter and gene annotations alongside sample-specific raw counts. The following cells inspect the table.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "merged_promoter_raw_counts.csv")) 
print(len(df))
df.head()

## Load and Inspect Sample Metadata

Load the metadata table linking each sample's `Run` ID to its cell type. This mapping is used to select samples and define the cell-type comparisons in the analysis.

In [ ]:
metdf = pd.read_csv(os.path.join(DATA_DIR, 'Metadata_Finalversion.csv'))

In [ ]:
metdf['Cell'] = metdf['Cell'].str.strip()

In [ ]:
metdf['Cell'].value_counts()

## Standardize Cell-Type Labels and Select Samples

Standardize the K1 label so equivalent samples are grouped consistently, then restrict the metadata to K1, S, DG44, and DXB11. These are the cell types included in the analysis.

In [ ]:
metdf['Cell'] = metdf['Cell'].str.replace('K1 (GS)', 'K1')

In [ ]:
# Retain only rows with Cell values: K1, S, DG44, DXB11 
cell_types_to_keep = ['K1', 'S', 'DG44', 'DXB11']
metdf = metdf[metdf['Cell'].isin(cell_types_to_keep)]

In [ ]:
metdf.head()

In [ ]:
def get_samples_by_metadata(metdf, column, value):
    return metdf[metdf[column] == value]['Run'].tolist()

## Retain Raw Counts for Samples in the Metadata

Keep the promoter annotation columns and the `.raw_counts` columns whose sample IDs appear in the metadata. This ensures that the count table and sample information refer to the same samples.

In [ ]:
metadata_cols = df.columns[:7]
samples_in_metdf = set(metdf["Run"].dropna().astype(str))

sample_cols = [
    col for col in df.columns[7:]
    if col.endswith(".raw_counts")
    and col.removesuffix(".raw_counts") in samples_in_metdf
]

cols_to_keep = list(metadata_cols) + sample_cols
df = df.loc[:, cols_to_keep]

print(
    f"df now has {len(df.columns)} columns "
    f"({len(metadata_cols)} metadata + {len(sample_cols)} raw-count sample columns)"
)
print(f"Raw-count columns: {sample_cols}")

## Aggregate Promoter Counts to the Gene Level

Sum promoter raw counts by `geneId` to create a gene-level count table with one row per gene. This table is normalized separately and later used to compare promoter-level changes with overall gene expression.

In [ ]:
# Create gene_count_df: 1 row per geneId with aggregated counts
metadata_cols = df.columns[:7]
sample_cols = df.columns[7:]

# Step 2: Sum sample columns across ALL rows for each geneId
gene_counts_df = df.groupby('geneId')[sample_cols].sum()
gene_counts_df = gene_counts_df.reset_index()

In [ ]:
gene_counts_df.head(10)

# Preprocessing

## Normalize Promoter and Gene-Level Counts

Normalize promoter and gene counts separately using counts per million (CPM), then calculate `log2(CPM + 1)`. Promoter-level log-normalized values and gene-level log-normalized values are kept under distinct column suffixes.

### Define the Count-Normalization Function

Define a reusable function that calculates CPM and `log2(CPM + 1)` for each sample's raw-count column. The `log_suffix` argument allows the gene-level normalized columns to be named separately from the promoter-level columns.

In [ ]:
def normalize_raw_counts(data, raw_count_cols, log_suffix="lognorm"):
    """Add CPM-normalized and log2(CPM + 1) columns for raw-count samples."""
    result = data.copy()

    missing = [col for col in raw_count_cols if col not in result.columns]
    if missing:
        raise KeyError(f"Raw-count columns not found: {missing}")

    for raw_col in raw_count_cols:
        sample = raw_col.removesuffix(".raw_counts")
        library_size_millions = result[raw_col].sum() / 1_000_000

        if library_size_millions <= 0:
            raise ValueError(f"{raw_col} has a zero or negative library size.")

        norm_col = f"{sample}.norm_counts"
        log_col = f"{sample}.{log_suffix}"

        result[norm_col] = result[raw_col] / library_size_millions
        result[log_col] = np.log2(result[norm_col] + 1)

    return result

### Identify Raw-Count Columns

Identify the raw-count columns in the promoter-level and gene-level tables. These column lists are passed to the normalization function in the next steps.

In [ ]:
# Identify each table's raw-count columns
promoter_raw_cols = [
    col for col in df.columns if col.endswith(".raw_counts")
]
gene_raw_cols = [
    col for col in gene_counts_df.columns if col.endswith(".raw_counts")
]

### Normalize Promoter Counts and Calculate Gene-Level Sums

Normalize each promoter's raw counts. Then, for each sample, sum the promoter log-normalized values for rows sharing a `geneId`; these sums are stored in `.gene_sum` columns.

In [ ]:
# Promoter table: normalize, then sum promoter lognorm values per gene
df = normalize_raw_counts(df, promoter_raw_cols)

for raw_col in promoter_raw_cols:
    sample = raw_col.removesuffix(".raw_counts")
    log_col = f"{sample}.lognorm"
    df[f"{sample}.gene_sum"] = df.groupby("geneId")[log_col].transform("sum")

### Normalize Gene-Level Counts

Normalize the gene-level raw counts separately. The resulting log-normalized columns use the `.gene_lognorm` suffix to distinguish them from promoter-level `.lognorm` values.

In [ ]:
# Gene table: normalize only; name its log-normalized values distinctly
gene_counts_normalized = normalize_raw_counts(
    gene_counts_df,
    gene_raw_cols,
    log_suffix="gene_lognorm",
)

### Add Gene-Level Normalized Values to the Promoter Table

Merge each gene's `.gene_lognorm` values into the promoter table by `geneId`. Each gene-level value is repeated for that gene's promoter rows, allowing promoter expression to be considered alongside overall gene expression.

In [ ]:
# Merge only geneId and the gene-level lognorm columns into the promoter table
gene_lognorm_cols = [
    col for col in gene_counts_normalized.columns
    if col.endswith(".gene_lognorm")
]

if gene_counts_normalized["geneId"].duplicated().any():
    raise ValueError("gene_counts_df must have only one row per geneId for this merge.")

df = df.merge(
    gene_counts_normalized[["geneId"] + gene_lognorm_cols],
    on="geneId",
    how="left",
    validate="many_to_one",
)

### Group Columns by Sample

Reorder the table so metadata columns appear first, followed by each sample's raw counts and corresponding normalized values. This changes column order only; it does not change the data.

In [ ]:
def reorder_columns(data):
    """Group each sample's count and normalized columns together."""
    sample_suffixes = (
        ".raw_counts",
        ".norm_counts",
        ".lognorm",
        ".gene_lognorm",
        ".gene_sum",
    )

    sample_names = [
        col.removesuffix(".raw_counts")
        for col in data.columns
        if col.endswith(".raw_counts")
    ]

    sample_columns = {
        f"{sample}{suffix}"
        for sample in sample_names
        for suffix in sample_suffixes
    }

    # Keep metadata and other non-sample columns first.
    ordered_columns = [
        col for col in data.columns
        if col not in sample_columns
    ]

    # Group available columns by sample in the desired order.
    for sample in sample_names:
        ordered_columns.extend(
            f"{sample}{suffix}"
            for suffix in sample_suffixes
            if f"{sample}{suffix}" in data.columns
        )

    # Preserve any remaining columns not covered by the expected suffixes.
    already_ordered = set(ordered_columns)
    ordered_columns.extend(
        col for col in data.columns
        if col not in already_ordered
    )

    return data.loc[:, ordered_columns]


df = reorder_columns(df)

In [ ]:
df.head(10)

## Separate Single- and Multi-Promoter Genes

Separate genes represented by one promoter row from genes represented by multiple promoter rows. The alternative-promoter comparison below uses the multi-promoter subset (`df_multi`).

In [ ]:
# Separate df into single and multi-promoter genes
gene_counts = df['geneId'].value_counts()

# Get geneIds that appear only once and multiple times
single_geneids = gene_counts[gene_counts == 1].index
multi_geneids = gene_counts[gene_counts > 1].index

# Create separate dataframes
df_single = df[df['geneId'].isin(single_geneids)].copy()
df_multi = df[df['geneId'].isin(multi_geneids)].copy()

# Identify Alternative Promoter Expression

Compare each promoter's log-normalized expression between a test cell type and the K1 control. Promoter-level fold change and a Welch's t-test are calculated, p-values are adjusted using the Benjamini–Hochberg method, and promoter changes are evaluated alongside gene-level expression.

A promoter is labeled as a hit when `promoterFC > 1.8`, `geneFC < 1.2`, and adjusted p-value `< 0.05` (These promoters would not show up in regular differential gene expression analysis). Fold changes are calculated as test divided by control, with a small pseudocount added to the group means.

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

def identify_alternative_promters(df_multi, metdf, meta_column, control_type, test_type):
    """Compare promoter and gene expression between control and test samples."""
    control_samples = get_samples_by_metadata(metdf, meta_column, control_type)
    test_samples = get_samples_by_metadata(metdf, meta_column, test_type)

    # Select available promoter and gene-level log-normalized columns.
    lognorm_control = [
        f"{sample}.lognorm"
        for sample in control_samples
        if f"{sample}.lognorm" in df_multi.columns
    ]
    lognorm_test = [
        f"{sample}.lognorm"
        for sample in test_samples
        if f"{sample}.lognorm" in df_multi.columns
    ]
    gene_control = [
        f"{sample}.gene_lognorm"
        for sample in control_samples
        if f"{sample}.gene_lognorm" in df_multi.columns
    ]
    gene_test = [
        f"{sample}.gene_lognorm"
        for sample in test_samples
        if f"{sample}.gene_lognorm" in df_multi.columns
    ]

    promoter_control_data = df_multi[lognorm_control].values
    promoter_test_data = df_multi[lognorm_test].values

    promoterFC = (
        (promoter_test_data.mean(axis=1) + 1e-6)
        / (promoter_control_data.mean(axis=1) + 1e-6)
    )

    # Run a Welch's t-test for each promoter.
    pvals = np.full(df_multi.shape[0], np.nan)

    for i in range(df_multi.shape[0]):
        control_values = promoter_control_data[i, :]
        test_values = promoter_test_data[i, :]

        # Retained from the original analysis to avoid zero-variance samples.
        pseudocount = 1e-07
        control_values_adj = control_values + np.random.normal(
            0, pseudocount, len(control_values)
        )
        test_values_adj = test_values + np.random.normal(
            0, pseudocount, len(test_values)
        )

        if np.std(control_values_adj) == 0 or np.std(test_values_adj) == 0:
            continue

        _, p_value = ttest_ind(
            test_values_adj,
            control_values_adj,
            equal_var=False,
        )
        pvals[i] = p_value

    # Adjust p-values for multiple testing.
    valid_pvals = ~np.isnan(pvals)
    adj_pvals = np.full_like(pvals, np.nan)
    adj_pvals[valid_pvals] = multipletests(
        pvals[valid_pvals],
        method="fdr_bh",
    )[1]

    # Calculate gene-level fold change.
    gene_control_data = df_multi[gene_control].values
    gene_test_data = df_multi[gene_test].values
    geneFC = (
        (gene_test_data.mean(axis=1) + 1e-6)
        / (gene_control_data.mean(axis=1) + 1e-6)
    )

    results = df_multi.copy()
    results["promoterFC"] = promoterFC
    results["geneFC"] = geneFC
    results["adj_pval"] = adj_pvals
    results["is_hit"] = (
        (results["promoterFC"] > 1.8)
        & (results["geneFC"] < 1.2)
        & (results["adj_pval"] < 0.05)
    )
    results["comparison"] = f"{test_type}_vs_{control_type}"

    return results

### Compare Each Cell Type Against the K1 Control

Run the alternative-promoter comparison for each cell type other than K1. The results are tagged with the test cell type and combined into `combined_results`.

In [ ]:
control = "K1"
celltypes = metdf["Cell"].unique()
all_results = []

for test_type in celltypes:
    if test_type == control:
        continue

    print(f"Comparing {test_type} to {control}...")

    result = identify_alternative_promters(
        df_multi,
        metdf,
        meta_column="Cell",
        control_type=control,
        test_type=test_type,
    )
    result["celltype"] = test_type
    all_results.append(result)

combined_results = pd.concat(all_results, ignore_index=True)

### Average Sample-Level Values by Cell Type

For each cell type, average the available sample-level columns across its samples. The resulting cell-type-level table is stored in `combined_results_cell`.

In [ ]:
# Map each cell type to its sample IDs.
celltype_to_samples = {
    celltype: metdf.loc[metdf["Cell"] == celltype, "Run"].astype(str).tolist()
    for celltype in metdf["Cell"].unique()
}
all_sample_ids = sorted(
    metdf["Run"].dropna().astype(str).unique(),
    key=len,
    reverse=True,
)

# Extract full suffixes, such as ".lognorm" and ".gene_lognorm".
col_suffixes = set()

for col in combined_results.columns:
    for sample in all_sample_ids:
        prefix = f"{sample}."
        if col.startswith(prefix):
            col_suffixes.add(col[len(sample):])
            break

# Keep metadata and result columns, then add cell-type averages.
metadata_cols = [
    col for col in combined_results.columns
    if not any(col.startswith(f"{sample}.") for sample in all_sample_ids)
]
combined_results_cell = combined_results[metadata_cols].copy()

for celltype in sorted(celltype_to_samples):
    samples = celltype_to_samples[celltype]

    for suffix in sorted(col_suffixes):
        sample_cols = [
            f"{sample}{suffix}"
            for sample in samples
            if f"{sample}{suffix}" in combined_results.columns
        ]

        if sample_cols:
            combined_results_cell[f"{celltype}{suffix}"] = (
                combined_results[sample_cols].mean(axis=1)
            )

print(
    f"combined_results_cell created with "
    f"{combined_results_cell.shape[0]} rows and "
    f"{combined_results_cell.shape[1]} columns"
)
print(f"Columns: {combined_results_cell.columns.tolist()[:10]}...")

In [ ]:
combined_results_cell.head(10)